# Task 4 - Trusted Profitability Model

Builds a star schema at **one row per payment attempt** and exports it for Power BI.

The rule that makes it trustworthy: every one-to-many source is collapsed to
transaction grain in its own staging frame **before** it touches the fact table.
Nothing is joined at its native grain.

Put this notebook in `05_profitability_model/`. It reads `../data/`.

In [1]:
"""Task 4 - trusted profitability model.

Builds a star schema at one row per payment ATTEMPT.

The rule that makes it trustworthy: every one-to-many source is collapsed to
transaction grain in its own staging frame BEFORE it touches the fact table.
Nothing is joined at its native grain.
"""
from pathlib import Path
import numpy as np
import pandas as pd

CHARGEBACK_HANDLING_FEE_USD = 15.0
RATE_TYPE = "SETTLEMENT"     # realised rate, not the quoted MID rate

DATA_DIR = "../data"
OUT_DIR = "model_output"

## Load

In [2]:
def load(data_dir):
    d = Path(data_dir)
    return dict(
        transaction=pd.read_csv(d / "transaction.csv", parse_dates=["created_at"]),
        payment_event=pd.read_csv(d / "payment_event.csv",
                                  parse_dates=["event_time", "ingestion_time"]),
        fraud_decision=pd.read_csv(d / "fraud_decision.csv"),
        settlement=pd.read_csv(d / "settlement.csv", parse_dates=["settlement_date"]),
        chargeback=pd.read_csv(d / "chargeback.csv", parse_dates=["opened_at", "resolved_at"]),
        fx_rate=pd.read_csv(d / "fx_rate.csv", parse_dates=["rate_date"]),
        route_cost=pd.read_csv(d / "route_cost.csv", parse_dates=["rate_date"]),
        pricing_plan=pd.read_csv(d / "pricing_plan.csv"),
        merchant=pd.read_csv(d / "merchant.csv"),
        customer=pd.read_csv(d / "customer.csv", parse_dates=["onboarding_date"]),
        merchant_risk_snapshot=pd.read_csv(d / "merchant_risk_snapshot.csv"),
    )

D = load(DATA_DIR)
for k, v in D.items():
    print(f"{k:<24} {len(v):>9,} rows")

transaction                260,287 rows
payment_event            1,000,316 rows
fraud_decision             466,451 rows
settlement                 224,242 rows
chargeback                     721 rows
fx_rate                      2,920 rows
route_cost                   3,285 rows
pricing_plan                     4 rows
merchant                       600 rows
customer                    15,000 rows
merchant_risk_snapshot       7,200 rows


## Staging

Each frame below states its grain in its docstring. All four collapse a
one-to-many source down to one row per transaction.

In [3]:
def stg_settlement(s):
    """GRAIN: 1 row per transaction_id. Collapses partial + correction pairs."""
    g = s.groupby("transaction_id", as_index=False).agg(
        merchant_fee_usd=("fee_amount", "sum"),
        settled_to_merchant_usd=("settlement_amount", "sum"),
        settlement_rows=("fee_amount", "size"),
        settlement_date=("settlement_date", "min"),
    )
    g["had_split_settlement"] = g.settlement_rows > 1
    return g


def stg_fraud(f):
    """GRAIN: 1 row per transaction_id. Normalises the two model score scales."""
    f = f.copy()
    # v3.0 emits 0-100; rescale so thresholds are comparable across versions
    f["risk_score_norm"] = np.where(f.risk_score > 1, f.risk_score / 100.0, f.risk_score)
    f["is_decline_rule"] = (f.decision == "DECLINE").astype("int8")
    g = f.groupby("transaction_id", as_index=False).agg(
        max_risk_score=("risk_score_norm", "max"),
        rules_fired=("rule_id", "size"),
        any_decline_rule=("is_decline_rule", "max"),
        model_version=("model_version", "max"),
    )
    g["any_decline_rule"] = g.any_decline_rule.astype(bool)
    return g


def stg_event(e):
    """GRAIN: 1 row per transaction_id. Pivots the lifecycle, keeps total latency."""
    g = e.groupby("transaction_id", as_index=False).agg(
        total_processing_ms=("processing_ms", "sum"),
        first_event_time=("event_time", "min"),
        last_event_time=("event_time", "max"),
        max_ingestion_time=("ingestion_time", "max"),
        event_count=("event_type", "size"),
    )
    g["reporting_lag_days"] = (
        (g.max_ingestion_time - g.last_event_time).dt.total_seconds() / 86400).round(2)
    return g.drop(columns=["max_ingestion_time"])


def stg_chargeback(c):
    """GRAIN: 1 row per transaction_id."""
    return c.groupby("transaction_id", as_index=False).agg(
        chargeback_amount_local=("amount", "sum"),
        chargeback_count=("chargeback_id", "size"),
    )

for name, fn, src in [("settlement", stg_settlement, "settlement"),
                      ("fraud", stg_fraud, "fraud_decision"),
                      ("event", stg_event, "payment_event"),
                      ("chargeback", stg_chargeback, "chargeback")]:
    g = fn(D[src])
    print(f"stg_{name:<12} {len(D[src]):>9,} source rows -> {len(g):>9,} at transaction grain")

stg_settlement     224,242 source rows ->   222,908 at transaction grain
stg_fraud          466,451 source rows ->   260,287 at transaction grain
stg_event        1,000,316 source rows ->   260,287 at transaction grain
stg_chargeback         721 source rows ->       721 at transaction grain


## Dimensions

In [4]:
def build_dims(D):
    m = D["merchant"].copy()
    c = D["customer"].copy()
    r = D["route_cost"][["route_id", "provider", "region"]].drop_duplicates().reset_index(drop=True)

    dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
    dim_date = pd.DataFrame({"date_key": dates})
    dim_date["year"] = dim_date.date_key.dt.year
    dim_date["month"] = dim_date.date_key.dt.to_period("M").astype(str)
    dim_date["quarter"] = dim_date.date_key.dt.to_period("Q").astype(str)
    dim_date["month_name"] = dim_date.date_key.dt.strftime("%b %Y")
    dim_date["day_of_week"] = dim_date.date_key.dt.day_name()

    return {"dim_merchant": m, "dim_customer": c, "dim_route": r, "dim_date": dim_date,
            "dim_pricing_plan": D["pricing_plan"].copy()}

dims = build_dims(D)
for k, v in dims.items():
    print(f"{k:<20} {len(v):>7,} rows")

dim_merchant             600 rows
dim_customer          15,000 rows
dim_route                  9 rows
dim_date                 365 rows
dim_pricing_plan           4 rows


## Fact table

GRAIN: one row per payment attempt. 260,287 rows in, 260,287 rows out.

Currency conversion uses `rate_to_usd` as a multiplier at the `SETTLEMENT`
rate type on the transaction date. `SETTLEMENT` is the realised rate; `MID`
is the quoted rate and overstates revenue.

Settlement fees are attributed back to transactions by summing `fee_amount`
to transaction grain first, so the 1,334 split settlements contribute once.
Captured transactions with no settlement record keep an imputed fee from the
rate card and carry `revenue_imputed = True` - dropping them would bias the
most recent months, where unsettled transactions concentrate.

In [5]:
def build_fact(D):
    t = D["transaction"].copy()
    t["txn_date"] = t.created_at.dt.normalize()
    t["txn_month"] = t.created_at.dt.to_period("M").astype(str)

    # --- data quality flags, applying the Task 3 remediation decisions --------
    t["dq_missing_merchant"] = t.merchant_id.isna()
    t["dq_impossible_negative"] = (t.amount < 0) & (t.status == "CAPTURED")
    t["is_reversal"] = t.status == "REVERSED"
    # quarantined rows stay in the table but are excluded from every measure
    t["dq_quarantined"] = t.dq_impossible_negative

    # --- FX: one rate per currency per day, one explicitly chosen rate type ---
    fx = D["fx_rate"]
    fx_sel = fx[fx.rate_type == RATE_TYPE].set_index(["currency", "rate_date"]).rate_to_usd
    key = pd.MultiIndex.from_arrays([t.currency, t.txn_date])
    t["fx_rate_to_usd"] = fx_sel.reindex(key).to_numpy()

    # constant-FX comparison uses each currency's first rate of the period
    opening = (fx[fx.rate_type == RATE_TYPE].sort_values("rate_date")
                 .groupby("currency").rate_to_usd.first())
    t["fx_rate_opening"] = t.currency.map(opening).to_numpy()

    t["gross_usd"] = t.amount * t.fx_rate_to_usd
    t["gross_usd_constant_fx"] = t.amount * t.fx_rate_opening

    # --- route cost: dated join, never on route_id alone ---------------------
    rc = D["route_cost"].set_index(["route_id", "rate_date"])
    rkey = pd.MultiIndex.from_arrays([t.route_id, t.txn_date])
    t["route_fixed_fee"] = rc.fixed_fee.reindex(rkey).to_numpy()
    t["route_variable_pct"] = rc.variable_fee_pct.reindex(rkey).to_numpy()
    t["provider"] = D["route_cost"][["route_id", "provider"]].drop_duplicates() \
                     .set_index("route_id").provider.reindex(t.route_id).to_numpy()

    # --- pre-aggregated children --------------------------------------------
    t = (t.merge(stg_settlement(D["settlement"]), on="transaction_id", how="left")
           .merge(stg_fraud(D["fraud_decision"]), on="transaction_id", how="left")
           .merge(stg_event(D["payment_event"]), on="transaction_id", how="left")
           .merge(stg_chargeback(D["chargeback"]), on="transaction_id", how="left"))

    # --- merchant attributes and as-of risk band ------------------------------
    m = D["merchant"].set_index("merchant_id")
    for col in ["category", "country", "pricing_plan", "risk_tier"]:
        t[f"merchant_{col}"] = m[col].reindex(t.merchant_id).to_numpy()

    rs = D["merchant_risk_snapshot"].set_index(["merchant_id", "snapshot_month"])
    skey = pd.MultiIndex.from_arrays([t.merchant_id, t.txn_month])
    t["merchant_risk_band"] = rs.risk_band.reindex(skey).to_numpy()

    cu = D["customer"].set_index("customer_id")
    t["customer_segment"] = cu.segment.reindex(t.customer_id).to_numpy()
    t["customer_country"] = cu.country.reindex(t.customer_id).to_numpy()

    # --- measures -------------------------------------------------------------
    t["is_attempt"] = 1
    t["is_success"] = (t.status == "CAPTURED") & ~t.dq_quarantined

    # revenue: the fee actually charged, or the rate card where no settlement
    # record has arrived yet. Dropping unsettled rows would bias recent months.
    pp = D["pricing_plan"].set_index("pricing_plan")
    rate = (t.merchant_pricing_plan.map(pp.rate_pct) / 100.0).to_numpy()
    fixed = t.merchant_pricing_plan.map(pp.fixed_fee_usd).to_numpy()
    imputed_rev = t.gross_usd.to_numpy() * rate + fixed

    # transactions with a missing merchant have no pricing plan, so the rate card
    # cannot be looked up. Fall back to the portfolio effective rate observed on
    # settled transactions, and flag it - leaving them NaN would silently drop
    # them from every weighted average downstream.
    settled = t.merchant_fee_usd.notna() & (t.gross_usd != 0)
    portfolio_rate = (t.loc[settled, "merchant_fee_usd"].sum()
                      / t.loc[settled, "gross_usd"].sum())
    t["revenue_fallback_rate"] = np.isnan(imputed_rev) & t.merchant_fee_usd.isna()
    imputed_rev = np.where(np.isnan(imputed_rev),
                           t.gross_usd.to_numpy() * portfolio_rate, imputed_rev)

    t["revenue_imputed"] = t.merchant_fee_usd.isna() & t.is_success
    t["merchant_revenue_usd"] = np.where(t.merchant_fee_usd.notna(),
                                         t.merchant_fee_usd.fillna(0), imputed_rev)

    t["processing_cost_usd"] = t.route_fixed_fee + t.route_variable_pct * t.gross_usd

    cb_usd = (t.chargeback_amount_local.fillna(0) * t.fx_rate_to_usd)
    t["chargeback_cost_usd"] = np.where(
        t.chargeback_count.fillna(0) > 0,
        cb_usd + CHARGEBACK_HANDLING_FEE_USD * t.chargeback_count.fillna(0), 0.0)

    for col in ["merchant_revenue_usd", "processing_cost_usd", "chargeback_cost_usd",
                "gross_usd", "gross_usd_constant_fx"]:
        t[col] = np.where(t.is_success, t[col], 0.0)

    t["contribution_usd"] = (t.merchant_revenue_usd
                             - t.processing_cost_usd
                             - t.chargeback_cost_usd)

    # FX impact: what translation alone cost, holding local pricing constant
    t["fx_impact_usd"] = np.where(
        t.is_success, (t.gross_usd - t.gross_usd_constant_fx) * rate, 0.0)

    cols = ["transaction_id", "txn_date", "txn_month", "customer_id", "merchant_id",
            "route_id", "provider", "channel", "status", "currency", "amount",
            "fx_rate_to_usd", "gross_usd", "gross_usd_constant_fx",
            "merchant_category", "merchant_country", "merchant_pricing_plan",
            "merchant_risk_tier", "merchant_risk_band",
            "customer_segment", "customer_country",
            "is_attempt", "is_success", "is_reversal",
            "merchant_revenue_usd", "processing_cost_usd", "chargeback_cost_usd",
            "contribution_usd", "fx_impact_usd",
            "route_fixed_fee", "route_variable_pct",
            "max_risk_score", "rules_fired", "any_decline_rule", "model_version",
            "total_processing_ms", "event_count", "reporting_lag_days",
            "settlement_rows", "had_split_settlement", "revenue_imputed",
            "revenue_fallback_rate",
            "chargeback_count",
            "dq_missing_merchant", "dq_impossible_negative", "dq_quarantined"]
    return t[cols]

fact = build_fact(D)
print("fact_transaction:", fact.shape)
print("contribution NaNs:", int(fact.contribution_usd.isna().sum()))
fact.head(3)

fact_transaction: (260287, 46)
contribution NaNs: 0


,transaction_id,txn_date,txn_month,customer_id,merchant_id,route_id,provider,channel,status,currency,...,event_count,reporting_lag_days,settlement_rows,had_split_settlement,revenue_imputed,revenue_fallback_rate,chargeback_count,dq_missing_merchant,dq_impossible_negative,dq_quarantined
0,TXN00000001,2025-01-01,2025-01,CUS002274,MER00458,RT_SG_03,PROV_NORTH,UPI,CAPTURED,SGD,...,4,0.00,1.0,False,False,False,NaN,False,False,False
1,TXN00000002,2025-01-01,2025-01,CUS004764,MER00347,RT_AE_03,PROV_NORTH,CARD,CAPTURED,AED,...,4,0.01,NaN,NaN,True,False,NaN,False,False,False
2,TXN00000003,2025-01-01,2025-01,CUS004274,MER00312,RT_AE_03,PROV_NORTH,BANK_TRANSFER,CAPTURED,AED,...,4,0.01,1.0,False,False,False,NaN,False,False,False


## Monthly KPIs

In [6]:
def monthly_kpis(fact):
    f = fact[~fact.dq_quarantined]
    g = f.groupby("txn_month").agg(
        attempted_volume=("is_attempt", "sum"),
        successful_volume=("is_success", "sum"),
        gross_payment_value_usd=("gross_usd", "sum"),
        merchant_revenue_usd=("merchant_revenue_usd", "sum"),
        processing_cost_usd=("processing_cost_usd", "sum"),
        chargeback_cost_usd=("chargeback_cost_usd", "sum"),
        fx_impact_usd=("fx_impact_usd", "sum"),
        contribution_usd=("contribution_usd", "sum"),
    ).reset_index()
    g["success_rate"] = (g.successful_volume / g.attempted_volume).round(4)
    g["contribution_per_successful_txn"] = (g.contribution_usd / g.successful_volume).round(4)
    g["effective_take_rate"] = (g.merchant_revenue_usd / g.gross_payment_value_usd).round(5)
    g["cost_per_successful_txn"] = (g.processing_cost_usd / g.successful_volume).round(4)
    return g


def validation_checks(D, fact):
    """Proof that the model does not double count."""
    rows = []
    t = D["transaction"]
    rows.append(("fact row count equals transaction row count",
                 len(fact), len(t), len(fact) == len(t)))
    rows.append(("fact transaction_id is unique",
                 fact.transaction_id.nunique(), len(fact),
                 fact.transaction_id.nunique() == len(fact)))

    cap = t[(t.status == "CAPTURED") & ~((t.amount < 0) & (t.status == "CAPTURED"))]
    rows.append(("successful volume equals captured count (quarantine excluded)",
                 int(fact.is_success.sum()), len(cap),
                 int(fact.is_success.sum()) == len(cap)))

    naive = t.merge(D["merchant_risk_snapshot"][["merchant_id", "snapshot_month"]],
                    on="merchant_id")
    rows.append(("naive risk-snapshot join would inflate row count",
                 len(naive), len(fact), len(naive) > len(fact)))

    # transaction side vs settlement side, on the same population:
    # settlement rows that match a transaction the model did not quarantine
    keep = set(fact.loc[~fact.dq_quarantined, "transaction_id"])
    s_side = D["settlement"][D["settlement"].transaction_id.isin(keep)].fee_amount.sum()
    t_side = fact.loc[fact.settlement_rows.notna() & ~fact.dq_quarantined,
                      "merchant_revenue_usd"].sum()
    rows.append(("settlement fees reconcile (transaction side vs settlement side)",
                 round(t_side, 2), round(s_side, 2), abs(t_side - s_side) < 1.0))

    dup = D["settlement"].groupby("transaction_id").size()
    dup_ids = set(dup[dup > 1].index)
    rows.append(("split settlements collapsed to one fact row",
                 int(fact.transaction_id.isin(dup_ids).sum()), len(dup_ids),
                 int(fact.transaction_id.isin(dup_ids).sum()) == len(dup_ids)))

    naive_fee = t.merge(D["settlement"][["transaction_id", "fee_amount"]],
                        on="transaction_id").fee_amount.sum()
    rows.append(("naive settlement join inflates fees (proof the pre-aggregation is needed)",
                 round(naive_fee, 2), round(s_side, 2), naive_fee > s_side))

    return pd.DataFrame(rows, columns=["check", "model_value", "source_value", "passed"])

kpis = monthly_kpis(fact)
pd.set_option("display.width", 240)
kpis[["txn_month", "attempted_volume", "successful_volume", "success_rate",
      "gross_payment_value_usd", "merchant_revenue_usd", "processing_cost_usd",
      "chargeback_cost_usd", "contribution_usd",
      "contribution_per_successful_txn", "effective_take_rate"]].round(3)

,txn_month,attempted_volume,successful_volume,success_rate,gross_payment_value_usd,merchant_revenue_usd,processing_cost_usd,chargeback_cost_usd,contribution_usd,contribution_per_successful_txn,effective_take_rate
0,2025-01,16380,14782,0.902,2955465.090,43473.144,15908.544,3717.674,23846.926,1.613,0.015
1,2025-02,17197,15481,0.900,3060586.920,45360.658,16675.290,3259.699,25425.669,1.642,0.015
2,2025-03,18027,16209,0.899,3113416.403,46027.286,16829.424,2325.133,26872.729,1.658,0.015
3,2025-04,18925,17016,0.899,3122948.857,46337.826,17091.313,2763.013,26483.499,1.556,0.015
4,2025-05,19862,17869,0.900,3225544.893,48055.874,17895.920,2640.054,27519.899,1.540,0.015
5,2025-06,20855,18794,0.901,3238322.633,48610.379,18134.437,3649.637,26826.305,1.427,0.015
6,2025-07,21876,19697,0.900,3407505.921,50847.440,20223.701,3275.179,27348.560,1.388,0.015
7,2025-08,22968,20317,0.885,3367309.414,50536.970,20439.718,2636.302,27460.950,1.352,0.015
8,2025-09,24103,21423,0.889,3412778.784,51491.557,21281.296,3131.316,27078.945,1.264,0.015
9,2025-10,25293,22746,0.899,3485371.771,52929.231,22275.801,4605.054,26048.375,1.145,0.015


In [7]:
a, b = kpis.iloc[0], kpis.iloc[-1]
print(f"attempted volume      {a.attempted_volume:>10,.0f} -> {b.attempted_volume:>10,.0f}   "
      f"{b.attempted_volume / a.attempted_volume - 1:+.1%}")
print(f"total contribution    {a.contribution_usd:>10,.0f} -> {b.contribution_usd:>10,.0f}   "
      f"{b.contribution_usd / a.contribution_usd - 1:+.1%}")
print(f"contribution per txn  {a.contribution_per_successful_txn:>10.3f} -> "
      f"{b.contribution_per_successful_txn:>10.3f}   "
      f"{b.contribution_per_successful_txn / a.contribution_per_successful_txn - 1:+.1%}")

attempted volume          16,380 ->     27,868   +70.1%
total contribution        23,847 ->     26,986   +13.2%
contribution per txn       1.613 ->      1.081   -33.0%


## Validation

The brief requires a reconciliation between transaction-side and settlement-side
totals, and a test proving a one-to-many join does not inflate payment amount.
Both are here.

In [8]:
checks = validation_checks(D, fact)
checks

,check,model_value,source_value,passed
0,fact row count equals transaction row count,260287.00,260287.0,True
1,fact transaction_id is unique,260287.00,260287.0,True
2,successful volume equals captured count (quara...,233208.00,233208.0,True
3,naive risk-snapshot join would inflate row count,3102228.00,260287.0,True
4,settlement fees reconcile (transaction side vs...,566298.30,566298.3,True
5,split settlements collapsed to one fact row,1334.00,1334.0,True
6,naive settlement join inflates fees (proof the...,567423.37,566298.3,True


## Export for Power BI

In [9]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

fact.to_csv(f"{OUT_DIR}/fact_transaction.csv", index=False)
for name, df in dims.items():
    df.to_csv(f"{OUT_DIR}/{name}.csv", index=False)
kpis.to_csv(f"{OUT_DIR}/kpi_monthly.csv", index=False)
checks.to_csv(f"{OUT_DIR}/validation_checks.csv", index=False)

for f in sorted(os.listdir(OUT_DIR)):
    print(f"{f:<30} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:>7.1f} MB")

05_metric_definitions.xlsx         0.0 MB
dim_customer.csv                   0.6 MB
dim_date.csv                       0.0 MB
dim_merchant.csv                   0.0 MB
dim_pricing_plan.csv               0.0 MB
dim_route.csv                      0.0 MB
fact_transaction.csv              90.4 MB
kpi_monthly.csv                    0.0 MB
validation_checks.csv              0.0 MB


## Metric definitions

Every KPI states numerator, denominator, date basis and exclusion rules, and is
marked additive or not. Power BI must not sum or average a non-additive measure
across segments.

In [10]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

metrics = pd.DataFrame([
 ("Attempted volume", "Count of transaction rows", "n/a", "created_at",
  "Quarantined rows excluded", "Additive"),
 ("Successful volume", "Count of rows with status CAPTURED", "n/a", "created_at",
  "REVERSED and quarantined rows excluded", "Additive"),
 ("Success rate", "Successful volume", "Attempted volume", "created_at",
  "Same population both sides", "NOT additive"),
 ("Gross payment value (USD)", "sum(amount * rate_to_usd)", "n/a", "created_at",
  "SETTLEMENT rate type; successful only", "Additive"),
 ("Merchant revenue (USD)", "sum(settlement fee at transaction grain), imputed from rate card where absent",
  "n/a", "created_at", "Successful only; revenue_imputed flags the imputed rows", "Additive"),
 ("Processing cost (USD)", "sum(route fixed_fee + variable_fee_pct * gross_usd)", "n/a",
  "created_at matched to route_cost.rate_date", "Successful only; dated fee lookup", "Additive"),
 ("Chargeback cost (USD)", "sum(chargeback amount in USD + 15.00 handling fee per case)",
  "n/a", "Parent transaction created_at", "Attributed to the transaction, not the dispute date", "Additive"),
 ("FX impact (USD)", "sum((gross_usd - gross_usd at opening rate) * contracted rate)",
  "n/a", "created_at", "Successful only", "Additive"),
 ("Contribution profit (USD)", "Merchant revenue - processing cost - chargeback cost",
  "n/a", "created_at", "Successful only", "Additive"),
 ("Contribution per successful transaction", "Contribution profit (USD)",
  "Successful volume", "created_at", "The executive KPI", "NOT additive"),
 ("Effective take rate", "Merchant revenue (USD)", "Gross payment value (USD)",
  "created_at", "Compare within pricing_plan, never blended", "NOT additive"),
 ("Cost per successful transaction", "Processing cost (USD)", "Successful volume",
  "created_at", "", "NOT additive"),
], columns=["Metric", "Numerator", "Denominator", "Date basis", "Exclusions / notes", "Additive?"])

grains = pd.DataFrame([
 ("fact_transaction", "1 row per payment ATTEMPT", "transaction_id", f"{len(fact):,}"),
 ("dim_merchant", "1 row per merchant", "merchant_id", f"{len(dims['dim_merchant']):,}"),
 ("dim_customer", "1 row per customer", "customer_id", f"{len(dims['dim_customer']):,}"),
 ("dim_route", "1 row per route", "route_id", f"{len(dims['dim_route']):,}"),
 ("dim_date", "1 row per calendar day", "date_key", f"{len(dims['dim_date']):,}"),
 ("dim_pricing_plan", "1 row per pricing plan", "pricing_plan", f"{len(dims['dim_pricing_plan']):,}"),
 ("stg_settlement", "1 row per transaction_id", "transaction_id", "collapses split settlements"),
 ("stg_fraud", "1 row per transaction_id", "transaction_id", "collapses 1-4 rule decisions"),
 ("stg_event", "1 row per transaction_id", "transaction_id", "collapses 2-4 lifecycle events"),
 ("stg_chargeback", "1 row per transaction_id", "transaction_id", "0 or 1 chargeback"),
], columns=["Table", "Grain statement", "Key", "Rows"])

NAVY, ARIAL = "1F3864", "Arial"
thin = Side(style="thin", color="BFBFBF"); box = Border(left=thin, right=thin, top=thin, bottom=thin)

def sheet(wb, title, df, widths):
    ws = wb.create_sheet(title)
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append(row)
    for c in range(1, df.shape[1] + 1):
        cell = ws.cell(row=1, column=c)
        cell.font = Font(name=ARIAL, bold=True, size=10, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor=NAVY)
        cell.alignment = Alignment(vertical="center", wrap_text=True); cell.border = box
    ws.row_dimensions[1].height = 30
    for r in range(2, df.shape[0] + 2):
        for c in range(1, df.shape[1] + 1):
            cell = ws.cell(row=r, column=c)
            cell.font = Font(name=ARIAL, size=9)
            cell.alignment = Alignment(vertical="top", wrap_text=True); cell.border = box
    for col, w in widths.items():
        ws.column_dimensions[col].width = w
    ws.freeze_panes = "A2"

wb = Workbook(); wb.remove(wb.active)
sheet(wb, "Metric Definitions", metrics, {"A": 38, "B": 52, "C": 22, "D": 30, "E": 42, "F": 14})
sheet(wb, "Grain Statements", grains, {"A": 22, "B": 34, "C": 20, "D": 28})
sheet(wb, "Validation", checks, {"A": 62, "B": 18, "C": 18, "D": 10})
wb.save(f"{OUT_DIR}/05_metric_definitions.xlsx")
print("wrote", f"{OUT_DIR}/05_metric_definitions.xlsx")

wrote model_output/05_metric_definitions.xlsx
